# Programme Services Exploratory Data Analysis

## Purpose

This examines how wellness services are assigned to programmes.

The analysis checks whether programme and service identifiers are complete, unique where required and also suitable for joining with the Programmes and Services datasets.

It also checks for missing relationships, duplicate assignments and records that reference programmes or services that do not exist.

In [17]:
from pathlib import Path
import pandas as pd

current_folder = Path.cwd()
project_root = next(
    folder for folder in [current_folder, *current_folder.parents]
    if (folder / "data" / "raw").exists()
)
data_folder = project_root / "data" / "raw"

sorted(file.name for file in data_folder.glob("*.csv"))

['Participations.csv',
 'Programme_Services.csv',
 'Programmes.csv',
 'Screenings.csv',
 'Services.csv']

In [18]:
programme_services = pd.read_csv(data_folder / "Programme_Services.csv")
programmes = pd.read_csv(data_folder / "Programmes.csv")
services = pd.read_csv(data_folder / "Services.csv")

print("Programme-service relationships:", programme_services.shape)
print("Programmes:", programmes.shape)
print("Services:", services.shape)


Programme-service relationships: (6, 5)
Programmes: (1, 11)
Services: (6, 6)


In [19]:
print("Programme Services columns:")
print(programme_services.columns.tolist())

print("\nProgrammes columns:")
print(programmes.columns.tolist())

print("\nServices columns:")
print(services.columns.tolist())

display(programme_services)

Programme Services columns:
['programme_service_id', 'programme_id', 'service_id', 'target_capacity', 'display_order']

Programmes columns:
['programme_id', 'organisation_id', 'branch_id', 'name', 'programme_type', 'start_date', 'end_date', 'venue', 'status', 'target_participants', 'created_at']

Services columns:
['service_id', 'code', 'name', 'category', 'default_unit', 'active']


,programme_service_id,programme_id,service_id,target_capacity,display_order
0,PS-001,PRG-001,SRV-BP,28,1
1,PS-002,PRG-001,SRV-BMI,28,2
2,PS-003,PRG-001,SRV-GLU,28,3
3,PS-004,PRG-001,SRV-CHOL,28,4
4,PS-005,PRG-001,SRV-STR,28,5
5,PS-006,PRG-001,SRV-FIT,28,6


In [20]:
print("Number of relationships:", len(programme_services))

print("\nMissing values:")
display(programme_services.isna().sum().to_frame("missing_count"))

duplicate_count = programme_services.duplicated(
    subset=["programme_id", "service_id"]
).sum()

print("Duplicate programme-service relationships:", duplicate_count)

Number of relationships: 6

Missing values:


,missing_count
programme_service_id,0
programme_id,0
service_id,0
target_capacity,0
display_order,0


Duplicate programme-service relationships: 0


In [21]:
invalid_programmes = programme_services[
    ~programme_services["programme_id"].isin(programmes["programme_id"])
]

invalid_services = programme_services[
    ~programme_services["service_id"].isin(services["service_id"])
]

print("Relationships with unknown programme IDs:", len(invalid_programmes))
print("Relationships with unknown service IDs:", len(invalid_services))

display(invalid_programmes)
display(invalid_services)

Relationships with unknown programme IDs: 0
Relationships with unknown service IDs: 0


,programme_service_id,programme_id,service_id,target_capacity,display_order


,programme_service_id,programme_id,service_id,target_capacity,display_order


In [22]:
joined_data = (
    programme_services
    .merge(
        programmes,
        on="programme_id",
        how="left",
        validate="many_to_one"
    )
    .merge(
        services,
        on="service_id",
        how="left",
        validate="many_to_one",
        suffixes=("_programme", "_service")
    )
)

print("Rows before joining:", len(programme_services))
print("Rows after joining:", len(joined_data))
print("Rows lost:", len(programme_services) - len(joined_data))

display(joined_data)

Rows before joining: 6
Rows after joining: 6
Rows lost: 0


,programme_service_id,programme_id,service_id,target_capacity,display_order,organisation_id,branch_id,name_programme,programme_type,start_date,end_date,venue,status,target_participants,created_at,code,name_service,category,default_unit,active
0,PS-001,PRG-001,SRV-BP,28,1,ORG-001,BR-001,2026 Workforce Wellness Programme,screening,2026-06-18T00:00:00Z,2026-06-18T00:00:00Z,Metsi Operations Site - Wellness Hall,completed,28,2026-05-22T10:00:00Z,blood_pressure,Blood Pressure Screening,screening,mmHg,True
1,PS-002,PRG-001,SRV-BMI,28,2,ORG-001,BR-001,2026 Workforce Wellness Programme,screening,2026-06-18T00:00:00Z,2026-06-18T00:00:00Z,Metsi Operations Site - Wellness Hall,completed,28,2026-05-22T10:00:00Z,bmi,BMI Assessment,screening,kg/m2,True
2,PS-003,PRG-001,SRV-GLU,28,3,ORG-001,BR-001,2026 Workforce Wellness Programme,screening,2026-06-18T00:00:00Z,2026-06-18T00:00:00Z,Metsi Operations Site - Wellness Hall,completed,28,2026-05-22T10:00:00Z,glucose,Blood Glucose Screening,screening,mmol/L,True
3,PS-004,PRG-001,SRV-CHOL,28,4,ORG-001,BR-001,2026 Workforce Wellness Programme,screening,2026-06-18T00:00:00Z,2026-06-18T00:00:00Z,Metsi Operations Site - Wellness Hall,completed,28,2026-05-22T10:00:00Z,cholesterol,Cholesterol Screening,screening,mmol/L,True
4,PS-005,PRG-001,SRV-STR,28,5,ORG-001,BR-001,2026 Workforce Wellness Programme,screening,2026-06-18T00:00:00Z,2026-06-18T00:00:00Z,Metsi Operations Site - Wellness Hall,completed,28,2026-05-22T10:00:00Z,stress,Stress Screening,screening,score,True
5,PS-006,PRG-001,SRV-FIT,28,6,ORG-001,BR-001,2026 Workforce Wellness Programme,screening,2026-06-18T00:00:00Z,2026-06-18T00:00:00Z,Metsi Operations Site - Wellness Hall,completed,28,2026-05-22T10:00:00Z,fitness,Fitness Assessment,fitness,score,True


In [23]:
services_per_programme = (
    programme_services
    .groupby("programme_id")
    .agg(service_count=("service_id", "nunique"))
    .reset_index()
    .sort_values("service_count", ascending=False)
)

programmes_per_service = (
    programme_services
    .groupby("service_id")
    .agg(programme_count=("programme_id", "nunique"))
    .reset_index()
    .sort_values("programme_count", ascending=False)
)

print("Services assigned to each programme:")
display(services_per_programme)

print("Programmes using each service:")
display(programmes_per_service)

Services assigned to each programme:


,programme_id,service_count
0,PRG-001,6


Programmes using each service:


,service_id,programme_count
0,SRV-BMI,1
1,SRV-BP,1
2,SRV-CHOL,1
3,SRV-FIT,1
4,SRV-GLU,1
5,SRV-STR,1


In [24]:
unassigned_programmes = programmes[
    ~programmes["programme_id"].isin(programme_services["programme_id"])
]

unused_services = services[
    ~services["service_id"].isin(programme_services["service_id"])
]

print("Programmes with no assigned services:", len(unassigned_programmes))
print("Services not used by any programme:", len(unused_services))

display(unassigned_programmes)
display(unused_services)

Programmes with no assigned services: 0
Services not used by any programme: 0


,programme_id,organisation_id,branch_id,name,programme_type,start_date,end_date,venue,status,target_participants,created_at


,service_id,code,name,category,default_unit,active


## Conclusion

The programme-service data is working correctly.

Each row connects a programme to a wellness service. There are no missing identifiers, and the same service has not been added to the same programme more than once.

Every programme ID points to a programme that exists. Every service ID also points to a service that exists. When the datasets were joined, no records were lost.

Every programme has at least one assigned service, and every available service is currently used by at least one programme.

This means `programme_id` and `service_id` are suitable for connecting programme data to service definitions and downstream application data.

The app can use this structure to show which services belong to a programme. It can also use it to calculate the number of services offered in each programme and identify how often each service is used.

The current data passes all the checks performed in this notebook. However, these checks should continue to run whenever new programmes or services are added. This will prevent broken references and duplicate service assignments from entering the system.

**Overall result:** `Programme_Services.csv` is complete, consistent and ready to be joined with `Programmes.csv` and `Services.csv`.